# 1.Runnable重试机制

知识点讲解：with_retry() 重试机制降低程序错误率

1. 组件定位：什么是 with_retry 重试机制
   - 在 LangChain 中，针对 Runnable 抛出的异常提供了重试机制——with_retry()
   - 当 Runnable 组件出现异常时，支持针对特定的异常或所有异常，重试特定的次数，
     并且配置每次重试的时间指数增加
   - 返回一个新的 Runnable（RunnableRetry），仍实现 Runnable 协议，可参与 | 管道组合
   - 与回退机制（with_fallbacks）并列，是 LLM 应用容错的两条核心路径之一

2. 为什么需要重试机制
   - LLM 应用强依赖网络与第三方 API，存在不可避免的瞬时故障
   - 常见瞬时错误：网络超时、限流（429）、服务端 5xx、连接重置
   - 这类错误往往重试一次即可成功，重试机制能显著提升链的成功率

3. with_retry() 的核心参数
   - stop_after_attempt: int - 最大尝试次数（含首次），默认 3
   - retry_if_exception_type: tuple - 需要重试的异常类型元组，默认 (Exception,)
   - wait_exponential_jitter: bool - 是否启用指数退避加随机抖动，默认 True
   - exponential_jitter_params: dict - 退避参数（initial/max/jitter 等）

4. stop_after_attempt 的准确语义
   - 该值表示"总尝试次数"，而不是"失败后额外重试的次数"
   - stop_after_attempt=2 → 最多执行 2 次（首次 + 1 次重试）
   - 若全部尝试都失败，抛出最后一次的原始异常

5. 指数退避与随机抖动（Exponential Backoff with Jitter）
   - 指数退避：每次重试的等待时间按指数增长（如 1s、2s、4s）
   - 随机抖动：在等待时间上叠加随机量，避免多客户端同时重试造成雪崩
   - 这是分布式系统中处理限流与瞬时故障的标准实践
   - wait_exponential_jitter=True 时，每次重试时间指数增加并随机再增加 1 秒内的时间

6. 底层实现原理
   - 运行原理非常简单，通过构建一个新的 Runnable 组件
   - 在执行调用类的函数时，循环特定次数，直到组件能正常执行结束即暂停
   - 在每次循环的过程中，休眠特定的时间（指数退避）
   - 若达到最大尝试次数仍失败，则抛出最后一次捕获的异常

7. 本示例的执行过程分析
   - counter 初始为 -1，全局变量在多次调用间持续累加
   - 第 1 次尝试：counter 变为 0，执行 2 / 0 → 抛出 ZeroDivisionError
   - 框架捕获异常，等待退避时间后进行第 2 次尝试
   - 第 2 次尝试：counter 变为 1，执行 2 / 1 = 2.0 → 成功返回
   - 最终输出 2.0，说明重试机制成功挽救了首次失败

8. 数据流转过程
   - 2（int）→ RunnableRetry(RunnableLambda(func), stop_after_attempt=2)
   - → 尝试 1：func(2) 抛出 ZeroDivisionError → 捕获并等待
   - → 尝试 2：func(2) 返回 2.0 → 成功
   - → 输出 2.0

9. 重试的适用边界与注意事项
   - 只应对瞬时错误重试；参数错误、认证失败等确定性错误重试无意义
   - 应通过 retry_if_exception_type 精确限定重试的异常类型
   - 被重试的函数最好是幂等的，否则可能产生重复副作用（如重复下单）
   - 重试会放大延迟与 Token 成本，需在超时预算内合理设置次数

10. 重试与回退的配合使用
   - with_retry()：同一组件重试，解决瞬时故障
   - with_fallbacks()：切换到备用组件，解决持久故障
   - 推荐组合：llm.with_retry(...).with_fallbacks([backup_llm])
   - 语义为：先对主模型重试若干次，全部失败后再切换备用模型


In [ ]:
from langchain_core.runnables import RunnableLambda
counter = -1

In [ ]:
def func(x):
    """模拟一个不稳定的函数：首次调用必然失败，重试后成功
    参数：
        x - 被除数，由 invoke 传入（本例为 2）
    返回：
        float - x 除以 counter 的结果
    异常：
        ZeroDivisionError - 当 counter 为 0 时（即首次调用）抛出
    设计说明：
        通过全局变量 counter 记录调用次数，模拟真实场景中的瞬时故障
        （如首次请求超时、重试后恢复正常）
    """
    global counter
    counter += 1
    print(f"当前的值为 {counter=}")
    return x / counter

In [ ]:
chain = RunnableLambda(func).with_retry(stop_after_attempt=2)
resp = chain.invoke(2)
print(resp)